In [1]:
# 1. Standard Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dbconfig import engine

print("Environment Ready.")

Environment Ready.


In [74]:
%%sql
with total_matches as
(select team1 as team, count(distinct match_id) as c
from ipl_master_view
group by team1

union all

select team2 as team, count(distinct match_id) as c
from ipl_master_view
group by team2)

select team, sum(c) as match_count
from total_matches
group by team
order by match_count desc;



,team,match_count
0,Mumbai Indians,261
1,Sunrisers Hyderabad,257
2,Royal Challengers Bengaluru,255
3,Delhi Capitals,252
4,Kolkata Knight Riders,251
5,Punjab Kings,246
6,Chennai Super Kings,238
7,Rajasthan Royals,221
8,Pune Warriors,46
9,Gujarat Titans,45


In [75]:
%%sql
select winner, count(distinct match_id) as win_count
from ipl_master_view
where winner is not null
group by winner
order by win_count desc;

,winner,win_count
0,Mumbai Indians,144
1,Chennai Super Kings,138
2,Kolkata Knight Riders,131
3,Royal Challengers Bengaluru,123
4,Sunrisers Hyderabad,117
5,Delhi Capitals,115
6,Punjab Kings,112
7,Rajasthan Royals,112
8,Gujarat Titans,28
9,Lucknow Super Giants,24


In [57]:
%%sql
select distinct result
from ipl_master_view;

,result
0,no result
1,runs
2,tie
3,wickets


In [63]:
%%sql
with runs_margin as (
select avg(result_margin) as r_m
from ipl_master_view
where result = 'runs'
and winner = 'Mumbai Indians'),

wickets_margin as (
select avg(result_margin) as w_m
from ipl_master_view
where result = 'wickets'
and winner = 'Mumbai Indians')

select r_m as runs_margin, w_m as wickets_margin
from runs_margin, wickets_margin;

,runs_margin,wickets_margin
0,32.241587,6.131718


In [88]:
result = match_count.merge(win_count, left_on='team', right_on='winner', how='left')
result = result.drop(columns=['winner'])
result['win_pct'] = (result['win_count'] / result['match_count'] * 100.0).round(2)
result

,team,match_count,win_count,win_pct
0,Mumbai Indians,261,144,55.17
1,Sunrisers Hyderabad,257,117,45.53
2,Royal Challengers Bengaluru,255,123,48.24
3,Delhi Capitals,252,115,45.63
4,Kolkata Knight Riders,251,131,52.19
5,Punjab Kings,246,112,45.53
6,Chennai Super Kings,238,138,57.98
7,Rajasthan Royals,221,112,50.68
8,Pune Warriors,46,12,26.09
9,Gujarat Titans,45,28,62.22


In [85]:
win_pct_top_3_played = result[result['team'].isin(['Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers Bengaluru'])].sort_values(by = 'win_pct', ascending = False)
win_pct_top_3_played

,team,match_count,win_count,win_pct
6,Chennai Super Kings,238,138,57.98
0,Mumbai Indians,261,144,55.17
2,Royal Challengers Bengaluru,255,123,48.24


In [11]:
%%sql
select count(*)
from matches
where toss_winner = winner

union all

select count(*)
from matches
where toss_winner != winner

union all

select count(*)
from matches;

,count
0,554
1,536
2,1095


In [13]:
%%sql
select toss_decision, count(*)
from matches
where toss_winner = winner
group by toss_decision;

,toss_decision,count
0,bat,177
1,field,377


In [29]:
%%sql
select venue,
       count(distinct id) as match_count,
       sum(case when team1 = winner then 1 else 0 end) as team1_wins,
       round(100.0 * sum(case when team1 = winner then 1 else 0 end) / count(distinct id), 2) as team1_win_pct
from matches
group by venue
order by match_count desc
limit 5;

,venue,match_count,team1_wins,team1_win_pct
0,Wankhede Stadium,118,59,50.00
1,M Chinnaswamy Stadium,94,46,48.94
2,Eden Gardens,93,53,56.99
3,MA Chidambaram Stadium,85,52,61.18
4,"Rajiv Gandhi International Stadium, Uppal",62,31,50.00


In [40]:
%%sql
select
    extract(year from to_date(date, 'YYYY-MM-DD')) as season,
    count(distinct id) as match_count,
    sum(case when winner = 'Chennai Super Kings' then 1 else 0 end) as win_count,
    round(100.0 * sum(case when winner = 'Chennai Super Kings' then 1 else 0 end)/ count(distinct id), 2) as win_pct
from matches
where team1 = 'Chennai Super Kings' or team2 = 'Chennai Super Kings'
group by extract(year from to_date(date, 'YYYY-MM-DD'))
order by season;

,season,match_count,win_count,win_pct
0,2008,16,9,56.25
1,2009,14,8,57.14
2,2010,16,9,56.25
3,2011,16,11,68.75
4,2012,18,10,55.56
5,2013,18,12,66.67
6,2014,16,10,62.50
7,2015,17,10,58.82
8,2018,16,11,68.75
9,2019,17,10,58.82


In [42]:
%%sql
select
    sum(case when winner = 'Chennai Super Kings' then 1 else 0 end) as win_count_csk,
    sum(case when winner = 'Mumbai Indians' then 1 else 0 end) as win_count_mi

from matches
where team1 in ('Chennai Super Kings', 'Mumbai Indians') and team2 in ('Chennai Super Kings', 'Mumbai Indians');

,win_count_csk,win_count_mi
0,17,20
